In [ ]:
from dotenv import load_dotenv
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END, add_messages
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from  langgraph.prebuilt import ToolNode, tools_condition

load_dotenv()

In [ ]:
# initialize the LLM
# parallel_tool_calls=False -- forces one tool call at a time; works around small model parallel call issues
llm = init_chat_model("llama-3.1-8b-instant", model_provider="groq")


In [ ]:
# mock external API
@tool
def get_stock_price(stock_symbol: str) -> float:
    """Returns the current stock price for the given stock symbol. In a real implementation, this would call an external API to get live stock prices.
    :param stock_symbol: The stock symbol to get the price for (e.g., "AAPL" for Apple Inc.)
    :return: The current stock price as a float
    """
        
    mock_prices = {
        "AAPL": 100.0,
        "GOOGL": 200.0,
        "AMZN": 300.0,
        "MSFT": 400.0,
    }
    return mock_prices.get(stock_symbol.upper(), 100.0)

tools= [get_stock_price]

# parallel_tool_calls=False -- forces one tool call at a time; works around small model parallel call issues
llm_with_tools = llm.bind_tools(tools, parallel_tool_calls=False)


In [ ]:
# graph state
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

# graph node function
def chatbot(state: ChatState)-> ChatState:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(ChatState)
builder.add_node("chatbot_node", chatbot)
builder.add_node("tools", ToolNode(tools))  # node must be named "tools" -- tools_condition routes to "tools" by default

builder.add_edge(START, "chatbot_node")
builder.add_conditional_edges("chatbot_node", tools_condition)  # routes to "tools" or END
# builder.add_edge("chatbot_node", END)  

graph = builder.compile()
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# one time tool call case
msg = {"role":"user", "content": "what is the price of GOOGL stock?"}
respState = graph.invoke({"messages": [msg]}) # info : inspect the respState 'messages' to see the tool call and response
respState["messages"]

In [ ]:
# tool call agentic fail - follow up question after tool call
msg1 = {"role":"user", "content": "what is the sum of GOOGL stock and AAPL stock price?"}
respState = graph.invoke({"messages": [msg1]}) 
respState["messages"]

In [ ]:
# Add edge loop
builder.add_edge("tools", "chatbot_node")  
graph = builder.compile()

from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Agentic loop case - follow up question after tool call
# official doc : https://docs.langchain.com/oss/python/langgraph/workflows-agents#agents
msg2 = {"role":"user", "content": "what is the sum of GOOGL stock and AAPL stock price?"}
respState = graph.invoke({"messages": [msg2]}) 
respState["messages"]